In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load and preprocess data
file_path = '../../JBI100 Data (2025-2026)/Hospital Beds Management/staff_schedule.csv'
df = pd.read_csv(file_path)
df.columns = [c.strip().lower().replace('#', '').replace(' ', '_') for c in df.columns]
df['week'] = df['week'].astype(int)
df['present'] = df['present'].astype(int)

# Dropdown widgets for filters
role_options = sorted(df['role'].unique())
service_options = sorted(df['service'].unique())
week_min, week_max = int(df['week'].min()), int(df['week'].max())

role_select = widgets.SelectMultiple(
    options=role_options,
    value=tuple(role_options),
    description='Role',
    disabled=False
)

service_select = widgets.SelectMultiple(
    options=service_options,
    value=tuple(service_options),
    description='Service',
    disabled=False
)

week_range_slider = widgets.IntRangeSlider(
    value=[week_min, week_max],
    min=week_min,
    max=week_max,
    step=1,
    description='Week',
    continuous_update=False
)

output_heatmap = widgets.Output()
output_role_bar = widgets.Output()
output_service_bar = widgets.Output()
output_detail = widgets.Output()

def filter_df(role_vals, service_vals, week_range):
    dff = df.copy()
    if role_vals:
        dff = dff[dff['role'].isin(role_vals)]
    if service_vals:
        dff = dff[dff['service'].isin(service_vals)]
    if week_range:
        w0, w1 = week_range
        dff = dff[(dff['week'] >= w0) & (dff['week'] <= w1)]
    return dff

def update_plots(change=None):
    roles = role_select.value
    services = service_select.value
    weeks = week_range_slider.value

    dff = filter_df(roles, services, weeks)

    with output_heatmap:
        clear_output(wait=True)
        heat_df = dff.groupby(['service', 'week'])['present'].mean().reset_index()
        if heat_df.empty:
            fig_heat = px.imshow([[0]], text_auto=True)
            fig_heat.update_layout(title='No Data')
        else:
            pivoted = heat_df.pivot(index='service', columns='week', values='present')
            fig_heat = px.imshow(pivoted, color_continuous_scale='Viridis', aspect='auto', origin='lower')
            fig_heat.update_layout(coloraxis_colorbar_title='Presence Rate',
                                   margin=dict(l=60, r=20, t=50, b=40))
        fig_heat.show()

    with output_role_bar:
        clear_output(wait=True)
        role_counts = (dff[dff['present'] == 1].groupby('role')['present']
                       .count().reset_index(name='count').sort_values('count'))
        fig_role = px.bar(role_counts, x='count', y='role', orientation='h', title='Present Staff by Role', text='count')
        fig_role.update_layout(margin=dict(l=20, r=20, t=40, b=20))
        fig_role.show()

    with output_service_bar:
        clear_output(wait=True)
        service_counts = (dff[dff['present'] == 1].groupby('service')['present']
                         .count().reset_index(name='count').sort_values('count'))
        fig_service = px.bar(service_counts, x='count', y='service', orientation='h', title='Present Staff by Service', text='count')
        fig_service.update_layout(margin=dict(l=20, r=20, t=40, b=20))
        fig_service.show()

# Attach update handlers
role_select.observe(update_plots, names='value')
service_select.observe(update_plots, names='value')
week_range_slider.observe(update_plots, names='value')

# Initial call to draw plots
update_plots()

# Display widgets and outputs

display(widgets.HBox([role_select, service_select, week_range_slider]))
display(output_heatmap)
display(output_role_bar)
display(output_service_bar)
display(output_detail)


Output()

Output()

Output()

Output()